## Структурная классификация белков

Хотя аминокислотная последовательность однозначно определяет трёхмерную структуру белка, предсказание полной 3D-структуры является сложной задачей.

В этом задании вам предстоит обучить несколько моделей для более простой задачи: предсказания типа белковой укладки по аминокислотной последовательности:

<style>
td, th {
   border: none!important;
}
</style>
| Orthogonal bundle | Beta-sandwich | Alpha-beta roll |
| :-------: | :------: | :----: |
|<img src="../assets/images/orthogonal_bundle.png" width="300"/>|<img src="../assets/images/beta_sandwich.png" width="300"/>|<img src="../assets/images/alpha_beta_roll.png" width="300"/>|


Датасет получен на основе базы данных [CATH](https://www.cathdb.info/wiki?id=data:index), представляющей иерархическую и стандартизированную классификацию белковых доменов по типам укладки. В датасете сделана подвыборка 9 архитектур укладки белков, для каждой собрано 2k примеров небольшой длины последовательности.

Данные и метки классов лежат в `assets/datasets/protein_fold`

In [1]:
from pathlib import Path

import pandas as pd
import torch
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

Класс датасета уже написан:

In [2]:
class CathSequencesDataset(Dataset):
    def __init__(self, subset_csv: Path, labels_csv: Path, add_cls_token: bool = False) -> None:
        # словарь: 20 аминокислот + 2 спец символа:
        # _ - pad token, добивает последовательность до нужной длины
        # ? — токен класса, он будет нужен для обучения трансформера
        vocab = "_?ACDEFGHIKLMNPQRSTVWY"
        df = pd.read_csv(subset_csv)
        labels = pd.read_csv(labels_csv)["architecture"]
        self.add_cls_token = add_cls_token
        self.vocab = {char: i for i, char in enumerate(vocab)}
        self.label_dict = {name: i for i, name in enumerate(labels)}
        self.sequences = [self._encode_sequence(s) for s in df["sequence"]]
        self.labels = [self.label_dict[label] for label in df["architecture"]]

    def __getitem__(self, index: int) -> tuple[list[int], int]:
        return self.sequences[index], self.labels[index]

    def __len__(self) -> int:
        return len(self.sequences)
    
    def _encode_sequence(self, sequence: str) -> list[int]:
        tokens = [self.vocab[char] for char in sequence]
        if self.add_cls_token:
            tokens = [1] + tokens
        return tokens

    @staticmethod
    def collate_fn(batch: list[tuple[list[int], int]]) -> tuple[Tensor, Tensor]:
        encoded, labels = zip(*batch)
        max_len = max(map(len, encoded))
        x = torch.zeros((len(encoded), max_len), dtype=int)
        for i, seq in enumerate(encoded):
            x[i, : len(seq)] = torch.tensor(seq)

        return x, torch.tensor(list(labels))

    def batch_decode(self, out_tokens: Tensor) -> list[str]:
        # декодирование всего батча токенов
        # сделано неточно, так как при получении токена окончания генерации (!)
        # остаток последовательности нужно отбросить
        inv_vocab = {v: k for k, v in self.vocab.items()}
        decoded_strings: list[str] = []
        for x in out_tokens:
            decoded_strings.append("".join([inv_vocab[i] for i in x.tolist() if i > 2]))

        return decoded_strings

In [3]:
rootdir = Path("../assets/datasets/protein_fold")
train_dataset = CathSequencesDataset(subset_csv=rootdir / "train.csv", labels_csv=rootdir / "labels.csv", add_cls_token=False)
tokens, label = train_dataset[0]
print("ID токенов:", tokens)
print("ID метки класса", label)

ID токенов: [14, 12, 19, 6, 11, 6, 14, 14, 10, 14, 10, 4, 18, 11, 12, 9, 17, 16, 18, 14, 5, 19, 18, 3, 19, 19, 19, 4, 19, 17, 4, 5, 4, 14, 5, 19, 10, 6, 13, 20, 21, 19, 4, 7, 19, 5, 19, 8, 13, 2, 10, 18, 10, 14, 16, 5, 5, 15, 21, 13, 2, 18, 21, 16, 19, 19, 17, 19, 11, 18, 19, 11, 8, 15, 4, 20, 11, 13, 7, 10, 5, 21, 10, 3, 10, 19, 17, 13, 10, 4, 11, 14, 2, 14, 9, 5, 10, 18, 9, 17]
ID метки класса 4


Последовательности в датасете имеют различную длину, поэтому для их объединения в один батч в `DataLoader(..., collate_fn=)` нужно будет передать фукнцию, которая правильно упаковывает список наблюдений в тензоры. Подходящая функция реализована в методе `CathSequencesDataset.collate_fn`

Посмотрим на мини-батч наблюдений:

In [4]:

train_dataset = CathSequencesDataset(subset_csv=rootdir / "train.csv", labels_csv=rootdir / "labels.csv", add_cls_token=False)
# sequences, labels = train_dataset.collate_fn([train_dataset[i] for i in range(4)])
train_loader = DataLoader(train_dataset, batch_size=4, collate_fn=train_dataset.collate_fn)
sequences, labels = next(iter(train_loader))
print("Последовательности:", sequences.shape)
print("Метки классов:", labels.shape)
sequences

Последовательности: torch.Size([4, 129])
Метки классов: torch.Size([4])


tensor([[14, 12, 19,  6, 11,  6, 14, 14, 10, 14, 10,  4, 18, 11, 12,  9, 17, 16,
         18, 14,  5, 19, 18,  3, 19, 19, 19,  4, 19, 17,  4,  5,  4, 14,  5, 19,
         10,  6, 13, 20, 21, 19,  4,  7, 19,  5, 19,  8, 13,  2, 10, 18, 10, 14,
         16,  5,  5, 15, 21, 13,  2, 18, 21, 16, 19, 19, 17, 19, 11, 18, 19, 11,
          8, 15,  4, 20, 11, 13,  7, 10,  5, 21, 10,  3, 10, 19, 17, 13, 10,  4,
         11, 14,  2, 14,  9,  5, 10, 18,  9, 17,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0],
        [ 7, 14, 11,  7, 17, 14,  5,  6, 18, 17, 12, 10, 14,  9, 13,  5,  4, 11,
          4,  5,  7,  9, 12, 19, 19, 21, 10, 16, 13,  9,  2,  7, 17,  7,  3, 15,
         11, 18,  6, 20,  5,  2, 17,  5, 16, 18,  9, 16, 17,  5,  2,  5,  4, 17,
         21,  8,  6, 17, 17,  2, 10, 12, 18,  2, 18,  6, 11, 17, 10, 10, 15,  5,
         19, 13, 12, 17,  4, 17,  2, 11,  4,  3, 19, 16,  4,  5,  2,  9, 13, 10,
      

### Задание 1 (5 баллов). Обучение классификатора на основе `nn.GRU`

Реализуйте классификатор последовательностей на основе GRU.
Ваш модуль будет иметь три нейросетевых блока:
1. Получение эмбеддингов для токенов
2. Получение скрытых состояний из рекуррентной сети
3. Классификатор последовательности на основе последнего скрытого состояния

В этом задании добейтесь точности классификации на валидационной выборке > 50%, для этого должно быть достаточно обучения в 15-20 эпох.

In [5]:
from torch import Tensor, nn


class GRUClassifier(nn.Module):
    """Модель для классификации"""

    def __init__(self, vocab_size: int, hidden_dim: int, n_classes: int) -> None:
        super().__init__()
        # эмбеддинги для аминокислот
        self.embed = nn.Embedding(vocab_size, hidden_dim, padding_idx=0)
        # рекуррентный модуль
        self.rnn = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,  # обратите внимание на этот параметр! Это частый источник ошибок при работе с рекуррентными сетями
        )
        # голова модели
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, tokens: Tensor) -> Tensor:
        # B — размер батча
        # L — длина последовательности
        # H — размерность эмбеддингов
        # C — число классов
        # V — размер словаря токенов

        # получаем эмбеддинги токенов: B x L -> B x L x H
        x = self.embed(tokens)
        # обрабатываем последовательность токенов с помощью RNN: B x L x H -> B x L x H
        states = self.rnn(x)
        # извлекаем последние скрытые состояния: B x L x H -> B x H
        last_state = states[0][:, -1, :]

        # используем их для классификации последовательностей: B x H -> B x C
        logits = self.classifier(last_state)

        return logits


In [6]:
rootdir = Path("../assets/datasets/protein_fold")
train_dataset = CathSequencesDataset(subset_csv=rootdir / "train.csv", labels_csv=rootdir / "labels.csv", add_cls_token=False)
val_dataset = CathSequencesDataset(subset_csv=rootdir / "val.csv", labels_csv=rootdir / "labels.csv", add_cls_token=False)

torch.manual_seed(42)

In [7]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=train_dataset.collate_fn)
val_loader = DataLoader(val_dataset, batch_size=4, collate_fn=train_dataset.collate_fn)

In [8]:
vocab = "_?ACDEFGHIKLMNPQRSTVWY"
vocab_size = len(vocab)
hidden_dim = 64
num_classes = 9

model = GRUClassifier(vocab_size, hidden_dim, num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [9]:
for epoch in range(20):
    model.train()
    train_loss = 0
    for tokens, labels in train_loader:
        tokens, labels = tokens.to(device), labels.to(device)
        logits = model.forward(tokens)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for tokens, labels in val_loader:
            tokens, labels = tokens.to(device), labels.to(device)
            logits = model.forward(tokens)

            preds = logits.argmax(dim=1)
            val_loss += F.cross_entropy(logits, labels).item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    avg_val_loss = val_loss / len(val_loader)
    acc = correct / total
    print(f"epoch {epoch}: train_loss = {avg_train_loss}, val_loss = {avg_val_loss}, accuracy = {acc}")

epoch 0: train_loss = 1.941083656847477, val_loss = 1.8621108021736146, accuracy = 0.31
epoch 1: train_loss = 1.6680937151014805, val_loss = 1.5483803300857544, accuracy = 0.41
epoch 2: train_loss = 1.5095595766976475, val_loss = 1.5397537281513214, accuracy = 0.426
epoch 3: train_loss = 1.3554537088945509, val_loss = 1.323493599653244, accuracy = 0.524
epoch 4: train_loss = 1.2429590605646372, val_loss = 1.2577296369075774, accuracy = 0.545
epoch 5: train_loss = 1.1587907633148133, val_loss = 1.192880593419075, accuracy = 0.564
epoch 6: train_loss = 1.0891703321691601, val_loss = 1.1818921984434128, accuracy = 0.573
epoch 7: train_loss = 1.0281571745257825, val_loss = 1.1328801736235619, accuracy = 0.594
epoch 8: train_loss = 0.9744370621042326, val_loss = 1.1378017409443855, accuracy = 0.593
epoch 9: train_loss = 0.9390682537490502, val_loss = 1.1358987007141113, accuracy = 0.578
epoch 10: train_loss = 0.9024419635729864, val_loss = 1.1411636237800122, accuracy = 0.58
epoch 11: train

### Задание 2 (5 баллов). Обучение классификатора на основе TransformerEncoder

Реализуйте классификатор последовательностей на основе блоков трансформера, приведённых в [notebooks/transformer_attention.ipynb](../notebooks/transformer_attention.ipynb).

Ваш модуль будет иметь три нейросетевых блока:
1. Получение начальных эмбеддингов для токенов и позиций
2. Преобразование эмбеддингов последовательностью блоков трансформера
3. Классификатор на основе финального эмбеддинга для токена <CLS>, который находится в начале каждой последовательности (не забудьте его включить в датасете!)

В этом задании добейтесь точности классификации на валидационной выборке > 60%, для этого должно быть достаточно обучения в 15-20 эпох.

In [10]:
import math


class PositionalEncoding(nn.Module):
    pe: Tensor

    def __init__(self, encoding_size: int, L: int = 10000, max_len: int = 1000):
        super().__init__()
        # можно было бы считать налету, но мы заготовим буфер под любую допустимую длину
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, encoding_size, 2) * (-math.log(L) / encoding_size)
        )
        pe = torch.zeros(max_len, encoding_size)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, t: Tensor) -> Tensor:
        return self.pe[t]

In [11]:
class TransformerEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int, pad_token_id: int = 0, max_position: int = 100):
        super().__init__()
        self.token_embeddings = nn.Embedding(vocab_size, hidden_size, padding_idx=pad_token_id)
        self.position_embeddings = PositionalEncoding(hidden_size, max_len=max_position)
        # self.position_embeddings = nn.Embedding(max_position, hidden_size)

        self.register_buffer(
            "position_ids", torch.arange(max_position).expand((1, -1)), persistent=False
        )

    def forward(self, input_ids: Tensor) -> torch.Tensor:
        B, N = input_ids.size()
        position_ids = self.position_ids[:, : N]
        inputs_embeds = self.token_embeddings(input_ids)
        position_embeddings = self.position_embeddings(position_ids)
        embeddings = inputs_embeds + position_embeddings
        return embeddings

In [12]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int) -> None:
        super().__init__()
        self.W_q = nn.Linear(input_dim, hidden_dim)
        self.W_k = nn.Linear(input_dim, hidden_dim)
        self.W_v = nn.Linear(input_dim, input_dim)

    def forward(self, x: Tensor) -> Tensor:
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)
        _, _, D_k = k.shape

        scores = torch.bmm(q, k.permute(0, 2, 1)) / D_k**0.5
        attention_weights = scores.softmax(-1)
        return torch.bmm(attention_weights, v)

In [13]:
class MultiHeadAttention(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, n_heads: int) -> None:
        super().__init__()
        self.heads = nn.ModuleList(
            [ScaledDotProductAttention(input_dim, hidden_dim) for i in range(n_heads)]
        )
        self.W_o = nn.Linear(n_heads * input_dim, input_dim)

    def forward(self, x: Tensor) -> Tensor:
        h = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.W_o(h)


In [14]:
class SelfAttentionLayer(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, n_heads: int) -> None:
        super().__init__()
        self.mha = MultiHeadAttention(input_dim, hidden_dim, n_heads)
        self.norm1 = nn.LayerNorm(input_dim)
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.ReLU(inplace=True),
            nn.Linear(input_dim, input_dim),
        )
        self.norm2 = nn.LayerNorm(input_dim)

    def forward(self, x: Tensor) -> Tensor:
        z = self.norm1(self.mha(x) + x)
        return self.norm2(self.mlp(z) + z)

In [15]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int, n_layers: int, n_classes: int = 9, max_position: int = 300, n_heads: int = 8):
        super().__init__()
        self.embeds = TransformerEmbeddings(vocab_size=vocab_size, hidden_size=hidden_size, max_position=max_position)  # эмбеддинги токенов + позиций
        self.trunc = nn.ModuleList([SelfAttentionLayer(hidden_size, hidden_size, n_heads) for i in range(n_layers)])  # последовательность SelfAttention модулей
        self.classifier = nn.Linear(hidden_size, n_classes)  # классификатор

    def forward(self, x: Tensor) -> Tensor:
        h = self.embeds.forward(x)
        for layer in self.trunc:
            h = layer.forward(h)

        cls_state = h[:, 0, :]

        logits = self.classifier(cls_state)

        return logits

In [16]:
rootdir = Path("../assets/datasets/protein_fold")
train_dataset_t = CathSequencesDataset(subset_csv=rootdir / "train.csv", labels_csv=rootdir / "labels.csv", add_cls_token=True)
val_dataset_t = CathSequencesDataset(subset_csv=rootdir / "val.csv", labels_csv=rootdir / "labels.csv", add_cls_token=True)

torch.manual_seed(42)

In [17]:
train_loader_t = DataLoader(train_dataset_t, batch_size=32, shuffle=True, collate_fn=train_dataset_t.collate_fn)
val_loader_t = DataLoader(val_dataset_t, batch_size=32, collate_fn=train_dataset_t.collate_fn)

In [18]:
vocab = "_?ACDEFGHIKLMNPQRSTVWY"
vocab_size = len(vocab)
hidden_dim = 128
num_classes = 9

model_t = TransformerEncoder(vocab_size=vocab_size, hidden_size=hidden_dim, n_layers=3, n_classes=num_classes, n_heads=8).to(device)
optimizer_t = torch.optim.AdamW(model_t.parameters(), lr=0.0005, weight_decay=0.1)


In [19]:
for epoch in range(15):
    model_t.train()
    train_loss = 0
    for tokens, labels in train_loader_t:
        tokens, labels = tokens.to(device), labels.to(device)
        logits = model_t.forward(tokens)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer_t.step()
        optimizer_t.zero_grad()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader_t)

    model_t.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for tokens, labels in val_loader_t:
            tokens, labels = tokens.to(device), labels.to(device)
            logits = model_t.forward(tokens)

            preds = logits.argmax(dim=1)
            val_loss += F.cross_entropy(logits, labels).item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    avg_val_loss = val_loss / len(val_loader_t)
    acc = correct / total
    print(f"epoch {epoch}: train_loss = {avg_train_loss}, val_loss = {avg_val_loss}, accuracy = {acc}")

epoch 0: train_loss = 1.4919595860242845, val_loss = 1.3813110329210758, accuracy = 0.502
epoch 1: train_loss = 1.300558342218399, val_loss = 1.2537527978420258, accuracy = 0.546
epoch 2: train_loss = 1.2363183180093764, val_loss = 1.2158400658518076, accuracy = 0.559
epoch 3: train_loss = 1.199041003704071, val_loss = 1.2751798555254936, accuracy = 0.556
epoch 4: train_loss = 1.1425639860630035, val_loss = 1.2229306790977716, accuracy = 0.574
epoch 5: train_loss = 1.0991924967765807, val_loss = 1.180701656267047, accuracy = 0.585
epoch 6: train_loss = 1.055383574485779, val_loss = 1.1318427622318268, accuracy = 0.596
epoch 7: train_loss = 1.0069603137969971, val_loss = 1.1602180525660515, accuracy = 0.596
epoch 8: train_loss = 0.9645218416452408, val_loss = 1.2144507486373186, accuracy = 0.585
epoch 9: train_loss = 0.9325337291955947, val_loss = 1.0759231839329004, accuracy = 0.625
epoch 10: train_loss = 0.8817353963255882, val_loss = 1.1283758543431759, accuracy = 0.619
epoch 11: tra

### Задание 3 (2 балла). Pre-norm vs Post-norm

Измените порядок следования механизма внимания, MLP и нормализаций на схему pre-normalized активаций (см. слайд **Постпроцессинг токенов: нормализация и MLP** в [slides/rnn_attention.pdf](../slides/rnn_attention.pdf)).

Запустите эксперимент с вашей лучшей конфигурацией модели из задания 2, заменив только определение `TransformerLayer`.

Изменилась ли точность или динамика обучения?

In [20]:
class PreNormSelfAttentionLayer(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, n_heads: int) -> None:
        super().__init__()
        self.mha = MultiHeadAttention(input_dim, hidden_dim, n_heads)
        self.norm1 = nn.LayerNorm(input_dim)
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.ReLU(inplace=True),
            nn.Linear(input_dim, input_dim),
        )
        self.norm2 = nn.LayerNorm(input_dim)

    def forward(self, x: Tensor) -> Tensor:
        z = x + self.mha(self.norm1(x))
        return z + self.mlp(self.norm2(z))

In [21]:
class PreNormTransformerEncoder(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int, n_layers: int, n_classes: int = 9, max_position: int = 300, n_heads: int = 8):
        super().__init__()
        self.embeds = TransformerEmbeddings(vocab_size=vocab_size, hidden_size=hidden_size, max_position=max_position)  # эмбеддинги токенов + позиций
        self.trunc = nn.ModuleList([PreNormSelfAttentionLayer(hidden_size, hidden_size, n_heads) for i in range(n_layers)])  # последовательность SelfAttention модулей
        self.classifier = nn.Linear(hidden_size, n_classes)  # классификатор

    def forward(self, x: Tensor) -> Tensor:
        h = self.embeds.forward(x)
        for layer in self.trunc:
            h = layer.forward(h)

        cls_state = h[:, 0, :]

        logits = self.classifier(cls_state)

        return logits

In [22]:
rootdir = Path("../assets/datasets/protein_fold")
train_dataset_prenorm = CathSequencesDataset(subset_csv=rootdir / "train.csv", labels_csv=rootdir / "labels.csv", add_cls_token=True)
val_dataset_prenorm = CathSequencesDataset(subset_csv=rootdir / "val.csv", labels_csv=rootdir / "labels.csv", add_cls_token=True)

torch.manual_seed(42)

In [23]:
train_loader_prenorm = DataLoader(train_dataset_prenorm, batch_size=32, shuffle=True, collate_fn=train_dataset_prenorm.collate_fn)
val_loader_prenorm = DataLoader(val_dataset_prenorm, batch_size=32, collate_fn=train_dataset_prenorm.collate_fn)

In [24]:
vocab = "_?ACDEFGHIKLMNPQRSTVWY"
vocab_size = len(vocab)
hidden_dim = 128
num_classes = 9

model_prenorm = PreNormTransformerEncoder(vocab_size=vocab_size, hidden_size=hidden_dim, n_layers=3, n_classes=num_classes, n_heads=8).to(device)
optimizer_prenorm = torch.optim.AdamW(model_prenorm.parameters(), lr=0.0005, weight_decay=0.1)


In [25]:
for epoch in range(15):
    model_prenorm.train()
    total_loss = 0
    for tokens, labels in train_loader_prenorm:
        tokens, labels = tokens.to(device), labels.to(device)
        logits = model_prenorm.forward(tokens)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer_prenorm.step()
        optimizer_prenorm.zero_grad()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader_prenorm)

    model_prenorm.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for tokens, labels in val_loader_prenorm:
            tokens, labels = tokens.to(device), labels.to(device)
            logits = model_prenorm.forward(tokens)

            preds = logits.argmax(dim=1)
            val_loss += F.cross_entropy(logits, labels).item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    avg_val_loss = val_loss / len(val_loader_prenorm)
    acc = correct / total
    print(f"epoch {epoch}: train_loss = {avg_train_loss}, val_loss = {avg_val_loss}, accuracy = {acc}")

epoch 0: train_loss = 1.488867285847664, val_loss = 1.3767453208565712, accuracy = 0.512
epoch 1: train_loss = 1.2923626492023468, val_loss = 1.2858296670019627, accuracy = 0.535
epoch 2: train_loss = 1.220232706308365, val_loss = 1.207712259143591, accuracy = 0.564
epoch 3: train_loss = 1.1750605832338332, val_loss = 1.2376274410635233, accuracy = 0.547
epoch 4: train_loss = 1.1041469572782516, val_loss = 1.1878144890069962, accuracy = 0.586
epoch 5: train_loss = 1.046747808098793, val_loss = 1.1245883535593748, accuracy = 0.592
epoch 6: train_loss = 1.0004714959859848, val_loss = 1.1104786284267902, accuracy = 0.605
epoch 7: train_loss = 0.9482792021036148, val_loss = 1.1146295592188835, accuracy = 0.609
epoch 8: train_loss = 0.8997318142652512, val_loss = 1.1479855626821518, accuracy = 0.604
epoch 9: train_loss = 0.8552704723477363, val_loss = 1.109072482213378, accuracy = 0.636
epoch 10: train_loss = 0.7951309002041816, val_loss = 1.1134541649371386, accuracy = 0.628
epoch 11: trai

Как видно из результатов и вывода значения метрики и ошибки в процессе обучения, pre-norm модель обучается чуть быстрее(на 6 эпохе преодолевает accuracy=0.6, в то время как post-norm - на 9й, а затем сильно колеблется вокруг этого значения), к тому же она обучается плавнее и стабильнее(меньшие колебания у значения accuracy, особенно после середины обучения), и pre-norm модель имеет меньший train_loss к концу обучения(0.6 против 0.71 у post-norm). При этом итоговые значения accuracy у pre-norm тоже оказались чуть лучше